In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import statsmodels.formula.api as smf

df = pd.read_csv("../data/fev1_clean.csv")
df["lh_c"]  = np.log(df["height"]) - np.log(1.5)
df["age10"] = (df["age"] - 12) / 10
df = df.sort_values(["id", "age"]).reset_index(drop=True)
print(df.shape)

(1994, 15)


In [3]:
# Hide each girl's last visit
n_visits  = df.groupby("id")["age"].transform("size")
is_last   = df.groupby("id").cumcount(ascending=False) == 0
test_mask = is_last & (n_visits >= 3)

train = df[~test_mask]
test  = df[test_mask].copy()
print(len(train), "training visits |", len(test), "hidden last visits")

#The idea: we pretend we don’t know each girl’s last measurement, and later check whether the model can predict it. Only girls with at least 3 visits are tested, so there’s some past to learn from.

1766 training visits | 228 hidden last visits


In [4]:
# Fit the final model on the trainig data only
mt = smf.mixedlm("log_fev1 ~ lh_c + bs(age, df=4)", data=train,
                 groups=train["id"], re_formula="~age10").fit(reml=False, method=["powell", "lbfgs"])
print("converged:", mt.converged)

#It’s the same model as m3 from Lesson 2, but it never sees the 228 hidden visits. That’s what makes the test honest: if the model could see them during fitting, predicting them would be cheating.

converged: True


/var/folders/wd/6lvzbdln5zs0vfn0nyhwwvdr0000gn/T/ipykernel_4224/3404356319.py:3: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  groups=train["id"], re_formula="~age10").fit(reml=False, method=["powell", "lbfgs"])


### 3 ways to predict the hidden visit
*a. Carry forward* (the naive baseline)
This simply guesses “same as her previous visit”. It ignores growth entirely, so any useful model must beat it.
.map(prev) looks up each test girl’s id in prev and fills in her previous FEV1.

*b. Population curve*
This predicts the average girl of her height and age. It uses only the fixed effects and knows nothing about this particular girl.
np.exp converts from log FEV1 back to litres.

*c. Personalised prediction*
This is the population curve plus her own adjustments, learned from her earlier visits:
* mt.random_effects is a dictionary: for each girl, her estimated intercept shift and slope shift
* lambda i: re[i].iloc[0] is a tiny one-line function: “given id i, take her first random effect (the level)”. .iloc[1] is the second one (the slope)
The slope shift is multiplied by age10, because a faster maturer drifts further from the average as she gets older.

In [13]:
print(df["age"].min(), df["age"].max())
# The model’s predictions are only trustworthy within ages 6–19.

formula = "log_fev1 ~ lh_c + bs(age, df=4, lower_bound=6, upper_bound=19)"
mt = smf.mixedlm(formula, data=train,
                 groups=train["id"], re_formula="~age10").fit(reml=False, method=["powell", "lbfgs"])
print("converged:", mt.converged)

6.4339 18.6913
converged: True


/var/folders/wd/6lvzbdln5zs0vfn0nyhwwvdr0000gn/T/ipykernel_4224/2676299144.py:5: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  groups=train["id"], re_formula="~age10").fit(reml=False, method=["powell", "lbfgs"])


In [14]:
# Carry forward
prev = train.groupby("id")["fev1"].last()
test["pred_carry"] = test["id"].map(prev)

In [15]:
# Population curve
test["pred_pop"] = np.exp(mt.predict(test))

In [16]:
# Personalised prediction
re = mt.random_effects
re_int   = test["id"].map(lambda i: re[i].iloc[0])   # her personal level
re_slope = test["id"].map(lambda i: re[i].iloc[1])   # her personal maturation speed
test["pred_girl"] = np.exp(mt.predict(test) + re_int + re_slope * test["age10"])

In [17]:
# Look at a few rows
test[["id", "age", "fev1", "pred_carry", "pred_pop", "pred_girl"]].head(10).round(2)

,id,age,fev1,pred_carry,pred_pop,pred_girl
6,1,16.37,2.39,2.44,2.60,2.64
14,2,17.63,3.11,2.52,2.77,2.74
23,3,16.02,2.96,3.02,2.94,3.09
33,4,17.79,3.28,2.82,3.02,2.92
40,5,17.52,2.30,2.52,2.74,2.60
51,6,16.98,3.25,3.43,3.41,3.43
58,7,17.49,2.67,2.80,3.33,2.82
67,8,18.21,3.24,3.19,3.14,3.16
76,9,15.01,2.64,2.72,2.78,2.82
86,10,17.03,3.23,3.05,2.36,3.30


In [ ]:
# Most hidden visits are at ages 15–18, on the plateau, where lungs barely grow. That makes carry forward a surprisingly strong baseline here. It’s a tough opponent, which makes the comparison more meaningful.

In [18]:
# Score the 3 methods with all rows (not the 10 first only)
for col, name in [("pred_carry", "Carry forward"), ("pred_pop", "Population curve"), ("pred_girl", "Personalised")]:
    err = test[col] - test["fev1"]
    rmse = np.sqrt((err ** 2).mean())
    print(f"{name:18s}  RMSE = {rmse:.3f} L   bias = {err.mean():+.3f} L")

Carry forward       RMSE = 0.266 L   bias = -0.076 L
Population curve    RMSE = 0.355 L   bias = -0.059 L
Personalised        RMSE = 0.186 L   bias = -0.039 L


### Summary
1. The personalised model wins clearly. Its errors are about 30% smaller than carry forward and almost half of the population curve’s.

2. The population curve is the worst, even worse than the naive baseline. Knowing who the girl is matters more than knowing her height and age. That matches your ICC of 0.73: most of the unexplained variation is “this girl is just generally higher or lower”.

3. How good is 0.186 L? At about 3 L, that’s roughly 6%. In Lesson 2, the visit-to-visit noise was about 6% too (√0.0033 ≈ 0.057 on the log scale). So the personalised predictions are almost as accurate as repeating the test itself. No model can predict noise, so there’s little room left to improve.

4. All three biases are slightly negative. Predictions are a little too low on average. For carry forward that makes sense, because girls still grow a bit between visits. For the models, it’s small (−0.04 L), but worth noting: late growth spurts, like girl 2’s, are hard to anticipate.

5. The clinical message: for a new child with no history, only the population curve is available. Each extra measurement makes predictions much better. That’s a concrete argument for longitudinal follow-up in cohorts like the one in this PhD.

In [19]:
# Is the difference real or luck
rng = np.random.default_rng(42)
diffs = []
for _ in range(2000):
    s = test.sample(len(test), replace=True, random_state=rng)
    rmse_carry = np.sqrt(((s["pred_carry"] - s["fev1"]) ** 2).mean())
    rmse_girl  = np.sqrt(((s["pred_girl"]  - s["fev1"]) ** 2).mean())
    diffs.append(rmse_carry - rmse_girl)

low, high = np.percentile(diffs, [2.5, 97.5])
print(f"Improvement of personalised over carry forward: {np.mean(diffs):.3f} L (95% CI {low:.3f} to {high:.3f})")

Improvement of personalised over carry forward: 0.078 L (95% CI 0.026 to 0.148)
